In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D 
from matplotlib.animation import FuncAnimation
from math import pi

plt.style.use("dark_background")

G   = 6.67430e-8
k_B = 1.380649e-16
m_H = 1.6735575e-24

pc   = 3.085677581e18
Msun = 1.98847e33
Lsun = 3.828e33

N_PARTICLES        = 1500
N_CLUMPS           = 300

BOX_SIZE_PC        = 1.5
GAS_TEMPERATURE_K  = 10.0
MU_MEAN            = 2.33
N_H2_CM3           = 1e3

TOTAL_STEPS        = 260
SUBSTEPS_PER_FRAME = 3
DT_YEARS           = 5000.0

SOFTENING_FRACTION = 0.02
SMOOTHING_FRACTION = 0.03
MIN_NEIGHBORS      = 30

STAR_MASS_FACTOR   = 12.0
VIEW_PC = 4.0 

BLOWOUT_COEFF = 9.5e-2

box_size_cm = BOX_SIZE_PC * pc
volume_cm3  = box_size_cm**3

rho0 = N_H2_CM3 * MU_MEAN * m_H
M_tot = rho0 * volume_cm3
m_particle = M_tot / N_PARTICLES

dt_s = DT_YEARS * 365.25 * 24 * 3600.0

softening = SOFTENING_FRACTION * box_size_cm
h_smooth  = SMOOTHING_FRACTION  * box_size_cm
kernel_volume = (4/3)*pi*h_smooth**3

rho_crit = rho0 * 550.0

print(f"Initial mean density = {rho0:.2e} g/cm^3")
print(f"Total mass ≈ {M_tot/Msun:.1f} Msun")
print(f"Particle mass ≈ {m_particle/Msun:.3f} Msun")
print(f"Collapse threshold = {rho_crit/rho0:.0f}× mean density")

np.random.seed(42)
positions = np.zeros((N_PARTICLES, 3))

weights = np.random.rand(N_CLUMPS)
weights /= weights.sum()
counts = np.floor(weights * N_PARTICLES).astype(int)
counts[-1] = N_PARTICLES - counts[:-1].sum()

idx = 0
for k in range(N_CLUMPS):
    n = counts[k]
    if n <= 0:
        continue
    
    center = np.random.uniform(-0.5*BOX_SIZE_PC, 0.5*BOX_SIZE_PC, 3) * pc
    R = np.random.uniform(0.10, 0.20) * pc

    positions[idx:idx+n] = center + np.random.normal(scale=R, size=(n,3))
    idx += n

velocities = np.random.normal(scale=4e2, size=(N_PARTICLES, 3))
velocities -= velocities.mean(axis=0)

mass_array = np.full(N_PARTICLES, m_particle)

is_star   = np.zeros(N_PARTICLES, dtype=bool)
star_mass = np.full(N_PARTICLES, np.nan)
star_L    = np.full(N_PARTICLES, np.nan)
star_Teff = np.full(N_PARTICLES, np.nan)

def compute_accelerations(pos, masses, eps):
    N = len(pos)
    acc = np.zeros_like(pos)
    for i in range(N):
        d = pos - pos[i]
        r2 = np.sum(d*d, axis=1) + eps**2
        inv_r3 = r2**(-1.5)
        inv_r3[i] = 0.0
        acc[i] = G * np.sum((d.T * (masses * inv_r3)).T, axis=0)
    return acc

def apply_radiation_pressure(pos, vel, is_star, Larr, coeff):
    star_idx = np.where(is_star)[0]
    if len(star_idx) == 0:
        return vel
    
    for i in star_idx:
        rvec = pos - pos[i]
        r2 = np.sum(rvec*rvec, axis=1)
        near = r2 < (0.30*pc)**2
        near[i] = False
        
        if np.any(near):
            direction = rvec[near] / np.sqrt(r2[near])[:,None]
            vel[near] += coeff * Larr[i] * direction
    
    return vel

def sample_mass_kroupa():
    r = np.random.random()
    if r < 0.70:
        return np.random.uniform(0.08, 0.6)
    elif r < 0.95:
        return np.random.uniform(0.6, 2.0)
    else:
        return np.random.uniform(2.0, 30.0)

def temperature_to_color(T):
    if T >= 30000: return "#6aa2ff"   
    if T >= 10000: return "#84a8ff"   
    if T >= 7500:  return "#b5c9ff"   
    if T >= 6000:  return "#ffeef0"   
    if T >= 5200:  return "#ffdbb0"   
    if T >= 3700:  return "#ffbb77"   
    return "#ff8877"

def update_star_formation(pos, is_star, M, L, T, masses):
    for i in range(len(pos)):
        if is_star[i]:
            continue
        
        d2 = np.sum((pos - pos[i])**2, axis=1)
        neigh = (~is_star) & (d2 < h_smooth**2)
        n_nb = np.count_nonzero(neigh)
        if n_nb < MIN_NEIGHBORS:
            continue
        
        rho_local = n_nb * m_particle / kernel_volume
        if rho_local > rho_crit:
            is_star[i] = True
            M_i = sample_mass_kroupa()

            M[i] = M_i
            L[i] = M_i**3.5
            T[i] = 5800 * M_i**0.5
            
            masses[i] = STAR_MASS_FACTOR * m_particle
    
    return is_star, M, L, T, masses

fig = plt.figure(figsize=(11,5))
fig.patch.set_facecolor("black")

ax3d = fig.add_subplot(1,2,1, projection="3d")
ax_hr = fig.add_subplot(1,2,2)

ax3d.set_facecolor("black")
ax_hr.set_facecolor("black")

half_box = BOX_SIZE_PC / 2

def init_plot():

    ax3d.set_xlim(-VIEW_PC, VIEW_PC)
    ax3d.set_ylim(-VIEW_PC, VIEW_PC)
    ax3d.set_zlim(-VIEW_PC, VIEW_PC)

    ax3d.set_title("Star-forming Region (3D)", color="white")
    ax3d.set_xlabel("x [pc]", color="white")
    ax3d.set_ylabel("y [pc]", color="white")
    ax3d.set_zlabel("z [pc]", color="white")
    ax3d.grid(False)

    ax_hr.set_title("HR Diagram", color="white")
    ax_hr.set_xlabel("log10 Teff", color="white")
    ax_hr.set_ylabel("log10 L/Lsun", color="white")
    ax_hr.invert_xaxis()
    ax_hr.set_xlim(4.8, 3.3)
    ax_hr.set_ylim(-2, 3)
    

def update(frame):
    global positions, velocities, is_star, star_mass, star_L, star_Teff, mass_array
    
    if frame > 0:
        for _ in range(SUBSTEPS_PER_FRAME):
            acc = compute_accelerations(positions, mass_array, softening)
            velocities += acc * dt_s
            
            velocities = apply_radiation_pressure(
                positions, velocities, is_star, star_L, BLOWOUT_COEFF
            )
            
            positions += velocities * dt_s
            
            is_star, star_mass, star_L, star_Teff, mass_array = update_star_formation(
                positions, is_star, star_mass, star_L, star_Teff, mass_array
            )

    ax3d.cla()
    ax3d.view_init(elev=30, azim=frame*0.8)
    ax3d.grid(False)

    pos_pc = positions / pc
    gas = ~is_star
    stars = is_star

    if np.any(gas):
        ax3d.scatter(
            pos_pc[gas,0], pos_pc[gas,1], pos_pc[gas,2],
            s=5, alpha=0.23, c="lightblue"
        )

    if np.any(stars):
        T = star_Teff[stars]
        M = star_mass[stars]
        col = [temperature_to_color(t) for t in T]
        sizes = 35*np.sqrt(M)
        ax3d.scatter(
            pos_pc[stars,0], pos_pc[stars,1], pos_pc[stars,2],
            s=sizes, c=col, alpha=0.95, edgecolor="none"
        )

    ax3d.set_facecolor("black")
    ax3d.set_xlim(-VIEW_PC, VIEW_PC)
    ax3d.set_ylim(-VIEW_PC, VIEW_PC)
    ax3d.set_zlim(-VIEW_PC, VIEW_PC)

    ax3d.set_title("Star-forming Region (3D)", color="white")

    ax_hr.cla()
    ax_hr.set_facecolor("black")
    ax_hr.set_title("HR Diagram", color="white")
    ax_hr.set_xlabel("log10 Teff", color="white")
    ax_hr.set_ylabel("log10 L/Lsun", color="white")
    ax_hr.invert_xaxis()
    ax_hr.set_xlim(4.8, 3.3)
    ax_hr.set_ylim(-2, 3)

    if np.any(stars):
        T = star_Teff[stars]
        L = star_L[stars]
        colors = [temperature_to_color(t) for t in T]
        ax_hr.scatter(np.log10(T), np.log10(L), c=colors, s=20)

    fig.suptitle(
        f"t = {frame*DT_YEARS*SUBSTEPS_PER_FRAME:,.0f} years",
        color="white"
    )
    return []

anim = FuncAnimation(fig, update, frames=TOTAL_STEPS, init_func=init_plot, blit=False)
anim.save("animation1.mp4", fps=30, dpi=300, bitrate=8000)

def classify(T):
    if T >= 30000: return "O"
    if T >= 10000: return "B"
    if T >= 7500:  return "A"
    if T >= 6000:  return "F"
    if T >= 5200:  return "G"
    if T >= 3700:  return "K"
    return "M"

types = [classify(t) for t in star_Teff[is_star]]
print("\nFinal star counts by type:")
for name in ["O","B","A","F","G","K","M"]:
    print(f"{name}: {types.count(name)}")
print(f"Total stars formed: {len(types)}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  
from matplotlib.animation import FuncAnimation
from math import pi

plt.style.use("dark_background")

G   = 6.67430e-8
k_B = 1.380649e-16
m_H = 1.6735575e-24

pc   = 3.085677581e18
Msun = 1.98847e33
Lsun = 3.828e33

N_PARTICLES        = 1500
N_CLUMPS           = 250

BOX_SIZE_PC        = 1.5
GAS_TEMPERATURE_K  = 10.0
MU_MEAN            = 2.33
N_H2_CM3           = 1e3

TOTAL_STEPS        = 260
SUBSTEPS_PER_FRAME = 3
DT_YEARS           = 5000.0

SOFTENING_FRACTION = 0.02
SMOOTHING_FRACTION = 0.03
MIN_NEIGHBORS      = 30

STAR_MASS_FACTOR   = 12.0

BLOWOUT_COEFF = 8e-3

box_size_cm = BOX_SIZE_PC * pc
volume_cm3  = box_size_cm**3

rho0 = N_H2_CM3 * MU_MEAN * m_H
M_tot = rho0 * volume_cm3
m_particle = M_tot / N_PARTICLES

dt_s = DT_YEARS * 365.25 * 24 * 3600.0

softening = SOFTENING_FRACTION * box_size_cm
h_smooth  = SMOOTHING_FRACTION  * box_size_cm
kernel_volume = (4/3)*pi*h_smooth**3

rho_crit = rho0 * 220.0

print(f"Initial mean density = {rho0:.2e} g/cm^3")
print(f"Total mass ≈ {M_tot/Msun:.1f} Msun")
print(f"Particle mass ≈ {m_particle/Msun:.3f} Msun")
print(f"Collapse threshold = {rho_crit/rho0:.0f}× mean density")

np.random.seed(42)
positions = np.zeros((N_PARTICLES, 3))

weights = np.random.rand(N_CLUMPS)
weights /= weights.sum()
counts = np.floor(weights * N_PARTICLES).astype(int)
counts[-1] = N_PARTICLES - counts[:-1].sum()

idx = 0
for k in range(N_CLUMPS):
    n = counts[k]
    if n <= 0:
        continue
    
    center = np.random.uniform(-0.5*BOX_SIZE_PC, 0.5*BOX_SIZE_PC, 3) * pc
    R = np.random.uniform(0.10, 0.20) * pc

    positions[idx:idx+n] = center + np.random.normal(scale=R, size=(n,3))
    idx += n

velocities = np.random.normal(scale=4e2, size=(N_PARTICLES, 3))
velocities -= velocities.mean(axis=0)

mass_array = np.full(N_PARTICLES, m_particle)

is_star   = np.zeros(N_PARTICLES, dtype=bool)
star_mass = np.full(N_PARTICLES, np.nan)
star_L    = np.full(N_PARTICLES, np.nan)
star_Teff = np.full(N_PARTICLES, np.nan)

def compute_accelerations(pos, masses, eps):
    N = len(pos)
    acc = np.zeros_like(pos)
    for i in range(N):
        d = pos - pos[i]
        r2 = np.sum(d*d, axis=1) + eps**2
        inv_r3 = r2**(-1.5)
        inv_r3[i] = 0.0
        acc[i] = G * np.sum((d.T * (masses * inv_r3)).T, axis=0)
    return acc

def apply_radiation_pressure(pos, vel, is_star, Larr, coeff):
    star_idx = np.where(is_star)[0]
    if len(star_idx) == 0:
        return vel
    
    for i in star_idx:
        rvec = pos - pos[i]
        r2 = np.sum(rvec*rvec, axis=1)
        near = r2 < (0.30*pc)**2
        near[i] = False
        
        if np.any(near):
            direction = rvec[near] / np.sqrt(r2[near])[:,None]
            vel[near] += coeff * Larr[i] * direction
    
    return vel

def sample_mass_kroupa():
    r = np.random.random()
    if r < 0.60:
        return np.random.uniform(0.08, 0.6)
    elif r < 0.90:
        return np.random.uniform(0.6, 2.0)
    else:
        return np.random.uniform(2.0, 30.0)

def temperature_to_color(T):
    if T >= 30000: return "#6aa2ff"   
    if T >= 10000: return "#84a8ff"   
    if T >= 7500:  return "#b5c9ff"   
    if T >= 6000:  return "#ffeef0"   
    if T >= 5200:  return "#ffdbb0"   
    if T >= 3700:  return "#ffbb77"   
    return "#ff8877"

def update_star_formation(pos, is_star, M, L, T, masses):
    for i in range(len(pos)):
        if is_star[i]:
            continue
        
        d2 = np.sum((pos - pos[i])**2, axis=1)
        neigh = (~is_star) & (d2 < h_smooth**2)
        n_nb = np.count_nonzero(neigh)
        if n_nb < MIN_NEIGHBORS:
            continue
        
        rho_local = n_nb * m_particle / kernel_volume
        if rho_local > rho_crit:
            is_star[i] = True
            M_i = sample_mass_kroupa()

            M[i] = M_i
            L[i] = M_i**3.5
            T[i] = 5800 * M_i**0.5
            
            masses[i] = STAR_MASS_FACTOR * m_particle
    
    return is_star, M, L, T, masses

fig = plt.figure(figsize=(11,5))
fig.patch.set_facecolor("black")

ax3d = fig.add_subplot(1,2,1, projection="3d")
ax_hr = fig.add_subplot(1,2,2)

ax3d.set_facecolor("black")
ax_hr.set_facecolor("black")

half_box = BOX_SIZE_PC / 2

def init_plot():
    ax3d.set_xlim(-half_box, half_box)
    ax3d.set_ylim(-half_box, half_box)
    ax3d.set_zlim(-half_box, half_box)
    ax3d.set_title("Star-forming Region (3D)", color="white")
    ax3d.set_xlabel("x [pc]", color="white")
    ax3d.set_ylabel("y [pc]", color="white")
    ax3d.set_zlabel("z [pc]", color="white")

    ax_hr.set_title("HR Diagram", color="white")
    ax_hr.set_xlabel("log10 Teff", color="white")
    ax_hr.set_ylabel("log10 L/Lsun", color="white")
    ax_hr.invert_xaxis()
    ax_hr.set_xlim(4.8, 3.3)
    ax_hr.set_ylim(-2, 3)

def update(frame):
    global positions, velocities, is_star, star_mass, star_L, star_Teff, mass_array
    
    if frame > 0:
        for _ in range(SUBSTEPS_PER_FRAME):
            acc = compute_accelerations(positions, mass_array, softening)
            velocities += acc * dt_s
            
            velocities = apply_radiation_pressure(
                positions, velocities, is_star, star_L, BLOWOUT_COEFF
            )
            
            positions += velocities * dt_s
            
            is_star, star_mass, star_L, star_Teff, mass_array = update_star_formation(
                positions, is_star, star_mass, star_L, star_Teff, mass_array
            )

    ax3d.cla()
    ax3d.view_init(elev=30, azim=frame*0.8)

    pos_pc = positions / pc
    gas = ~is_star
    stars = is_star

    if np.any(gas):
        ax3d.scatter(
            pos_pc[gas,0], pos_pc[gas,1], pos_pc[gas,2],
            s=5, alpha=0.23, c="lightblue"
        )

    if np.any(stars):
        T = star_Teff[stars]
        M = star_mass[stars]
        col = [temperature_to_color(t) for t in T]
        sizes = 35*np.sqrt(M)
        ax3d.scatter(
            pos_pc[stars,0], pos_pc[stars,1], pos_pc[stars,2],
            s=sizes, c=col, alpha=0.95, edgecolor="none"
        )

    ax3d.set_facecolor("black")
    ax3d.set_xlim(-half_box, half_box)
    ax3d.set_ylim(-half_box, half_box)
    ax3d.set_zlim(-half_box, half_box)
    ax3d.set_title("Star-forming Region (3D)", color="white")

    ax_hr.cla()
    ax_hr.set_facecolor("black")
    ax_hr.set_title("HR Diagram", color="white")
    ax_hr.set_xlabel("log10 Teff", color="white")
    ax_hr.set_ylabel("log10 L/Lsun", color="white")
    ax_hr.invert_xaxis()
    ax_hr.set_xlim(4.8, 3.3)
    ax_hr.set_ylim(-2, 3)

    if np.any(stars):
        T = star_Teff[stars]
        L = star_L[stars]
        colors = [temperature_to_color(t) for t in T]
        ax_hr.scatter(np.log10(T), np.log10(L), c=colors, s=20)

    fig.suptitle(
        f"t = {frame*DT_YEARS*SUBSTEPS_PER_FRAME:,.0f} years",
        color="white"
    )
    return []

anim = FuncAnimation(fig, update, frames=TOTAL_STEPS, init_func=init_plot, blit=False)
anim.save("animation2.mp4", fps=30, dpi=300, bitrate=8000)

def classify(T):
    if T >= 30000: return "O"
    if T >= 10000: return "B"
    if T >= 7500:  return "A"
    if T >= 6000:  return "F"
    if T >= 5200:  return "G"
    if T >= 3700:  return "K"
    return "M"

types = [classify(t) for t in star_Teff[is_star]]
print("\nFinal star counts by type:")
for name in ["O","B","A","F","G","K","M"]:
    print(f"{name}: {types.count(name)}")
print(f"Total stars formed: {len(types)}")
